# Entrenar el LoRA de Elena — ai-toolkit (Z-Image) en Colab

Entrena un **LoRA de Z-Image** con la **UI web de ai-toolkit** (Ostris), corriendo en Colab.
NO usa 100 GB: baja el modelo (~pocos GB) + deps. Necesita **GPU con ≥12 GB** (L4/A100 recomendado).

**Cómo va:**
1. *Entorno de ejecución → Cambiar tipo de entorno →* **L4** (o A100). T4 (16GB) puede andar en 512px.
2. *Ejecutar todo.* Al final imprime una **URL pública** + un **token**.
3. Abrís esa URL (es la UI de ai-toolkit), creás un job, apuntás el dataset y entrenás.
4. El `.safetensors` sale a tu Drive: `MyDrive/ai-toolkit/output/`.

> Settings recomendados para Z-Image (verificados): modelo **Z-Image Turbo**, **~2000 steps**,
> LR **0.0001–0.0002**, resolución **512, 768**, **Cache Latents ON**, **Transformer Offload 0%**,
> LoRA rank **8**, dataset **6–15 imágenes** (máx lado 768).


## 1) Montar Drive (dataset de entrada + LoRA de salida viven acá)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/ai-toolkit'
DATASET = os.path.join(BASE, 'datasets', 'elena')
OUTPUT  = os.path.join(BASE, 'output')
for d in [DATASET, OUTPUT]:
    os.makedirs(d, exist_ok=True)
print('Dataset (subí acá las imágenes + .txt):', DATASET)
print('Salida del LoRA:', OUTPUT)
print()
print('>> SUBÍ a la carpeta dataset las 20 imágenes elena_*.png y sus elena_*.txt')
print('   (están en tu PC en: Produccion visual/assets/elena/lora-dataset-v1/)')


## 2) Instalar Node 20 (la UI de ai-toolkit lo necesita)


In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs > /dev/null 2>&1
!node -v


## 3) Clonar ai-toolkit + instalar (torch cu128 + requirements). Tarda unos minutos.


In [ ]:
%cd /content
import os
if not os.path.isdir('/content/ai-toolkit'):
    !git clone https://github.com/ostris/ai-toolkit.git
%cd /content/ai-toolkit
!pip install -q torch==2.9.1 torchvision==0.24.1 torchaudio==2.9.1 --index-url https://download.pytorch.org/whl/cu128
!pip install -q -r requirements.txt
print('\nai-toolkit instalado')


## 4) Lanzar la UI + túnel público

Imprime la **URL** y el **token de acceso**. Abrí la URL, y si pide contraseña, pegá el token.
La primera vez la UI compila (1-2 min) antes de responder.


In [ ]:
import threading, time, socket, subprocess, os, re, secrets
%cd /content/ai-toolkit/ui

TOKEN = secrets.token_urlsafe(8)
os.environ['AI_TOOLKIT_AUTH'] = TOKEN

if not os.path.isfile('/usr/local/bin/cloudflared'):
    os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared')

def wait_port(p):
    while socket.socket().connect_ex(('127.0.0.1', p)) != 0:
        time.sleep(2)

def tunnel():
    wait_port(8675)
    proc = subprocess.Popen(
        ['cloudflared','tunnel','--no-autoupdate','--protocol','http2','--url','http://127.0.0.1:8675'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    shown = False
    for line in proc.stdout:
        m = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
        if m and not shown:
            shown = True
            print('\n\n' + '='*60)
            print('  UI de ai-toolkit (abrila en el navegador):')
            print('  ' + m.group(0))
            print('  TOKEN (si pide contraseña): ' + TOKEN)
            print('='*60 + '\n')

threading.Thread(target=tunnel, daemon=True).start()
!npm run build_and_start


## 5) Cómo configurar el job en la UI (una vez abierta)

1. **New Job / Training**.
2. **Model:** elegí **Z-Image Turbo** y dejá que baje los pesos (o pegá el model path si lo pide).
3. **Dataset path:** `/content/drive/MyDrive/ai-toolkit/datasets/elena`
4. **Trigger word:** `elenacs` (ya está en los captions).
5. **Settings:** steps **2000**, LR **0.0001**, resolución **512,768**, **Cache Latents ON**,
   **Transformer Offload 0%**, LoRA rank **8**, batch size **1**.
6. **Output folder:** `/content/drive/MyDrive/ai-toolkit/output`
7. **Start** y mirá los samples cada N steps. ~2000 steps ≈ 1–1.5 hs en L4.

Al terminar, el `.safetensors` queda en tu Drive → lo enchufás en **easy-comfy** (LoRA por URL o a mano).
